# 07 从零采集 MuJoCo LeRobot 示教数据

        这一节把 `5.language_env.ipynb` 的交互采集流程整理成可配置、可复核的版本。最终产物是一个包含双相机图像、6 维关节状态、7 维动作和红/蓝杯语言指令的 LeRobot 数据集。

        采集不是纯 GPU 任务，NVIDIA 或 AMD 设备都可以完成。真正需要保持一致的是 MuJoCo 场景、LeRobot 数据格式、控制频率、state/action 定义和成功判定。远端 Jupyter 无法稳定接收 MuJoCo viewer 键盘事件时，可以在有桌面的机器采集，再把数据目录同步到 AMD 训练机。


In [1]:

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    override = (
        os.environ.get("AMD_TOPIC_ROOT")
        or os.environ.get("NOTEBOOK_TOPIC_ROOT")
        or os.environ.get("TOPIC_ROOT")
    )
    roots = [Path(override).expanduser()] if override else []
    cwd = Path.cwd().resolve()
    roots.extend([cwd, *cwd.parents])

    candidates = []
    for root in roots:
        candidates.extend(
            [
                root,
                root / "16-专题组队学习" / "04-AMD-ROCm策略复刻专题",
                root / "04-AMD-ROCm策略复刻专题",
            ]
        )
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError(
        "找不到 AMD ROCm 专题目录。请从仓库根目录、专题目录启动 Jupyter，"
        "或设置 AMD_TOPIC_ROOT。"
    )


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(
    os.environ.get("PROJECT_ROOT", TOPIC_ROOT / "external" / "mujoco_pnp")
).expanduser()
DATA_ROOT = Path(os.environ.get("DATA_ROOT", TOPIC_ROOT / "data")).expanduser()
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


TOPIC_ROOT = $NOTEBOOK_TOPIC_ROOT
PROJECT_ROOT = $PROJECT_ROOT
DATA_ROOT = $PROJECT_ROOT
OUTPUT_ROOT = $OUTPUT_ROOT
MODEL_ROOT = $MODEL_ROOT


In [2]:

try:
    from IPython.display import Image, Markdown, Video, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"

    def Video(filename=None, embed=False, width=None, **kwargs):
        return f"[video] {filename}"


def show_video(filename, title=None, width=960):
    path = ASSET_DIR / filename
    if title:
        display(Markdown(f"**{title}**"))
    if not path.exists():
        print(f"缺少视频素材：{path}")
        return
    try:
        display(Video(filename=str(path), embed=True, width=width, html_attributes="controls muted"))
    except TypeError:
        display(Video(filename=str(path), embed=True, width=width))


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        if path.suffix.lower() in {".mp4", ".webm", ".mov", ".m4v"}:
            show_video(filename, width=width)
            return
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


In [3]:
import shlex

try:
    import yaml
except ImportError as exc:
    raise RuntimeError("当前环境缺少 PyYAML，请先执行 pip install pyyaml。") from exc


def require_project_layout():
    required = [
        PROJECT_ROOT / "train_model.py",
        PROJECT_ROOT / "env_config.py",
        PROJECT_ROOT / "asset" / "example_scene_y2.xml",
        PROJECT_ROOT / "mujoco_env" / "y_env2.py",
    ]
    missing = [path for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "PROJECT_ROOT 不是 04mujoco 教程目录，缺少：\n"
            + "\n".join(str(path) for path in missing)
        )
    return True


def dataset_report(dataset_root):
    dataset_root = Path(dataset_root)
    info_path = dataset_root / "meta" / "info.json"
    tasks_path = dataset_root / "meta" / "tasks.jsonl"
    if not info_path.exists():
        raise FileNotFoundError(f"找不到 LeRobot 元数据：{info_path}")
    info = json.loads(info_path.read_text(encoding="utf-8"))
    tasks = []
    if tasks_path.exists():
        tasks = [
            json.loads(line)["task"]
            for line in tasks_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
    features = info.get("features", {})
    rows = [
        ("repo_id", info.get("repo_id", "")),
        ("episodes", info.get("total_episodes", 0)),
        ("frames", info.get("total_frames", 0)),
        ("fps", info.get("fps", "")),
        ("state shape", features.get("observation.state", {}).get("shape")),
        ("action shape", features.get("action", {}).get("shape")),
        ("tasks", " / ".join(tasks)),
    ]
    md_table(["数据项", "读取结果"], rows)
    return info, tasks


def make_training_config(
    policy_type,
    dataset_repo_id,
    dataset_root,
    output_dir,
    steps,
    batch_size,
    chunk_size,
    n_action_steps,
    save_freq,
    seed=42,
):
    return {
        "dataset": {"repo_id": dataset_repo_id, "root": str(Path(dataset_root))},
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "save_checkpoint": True,
        "output_dir": str(Path(output_dir)),
        "batch_size": int(batch_size),
        "job_name": Path(output_dir).name,
        "resume": False,
        "seed": int(seed),
        "num_workers": 4,
        "steps": int(steps),
        "eval_freq": 0,
        "log_freq": max(1, min(50, int(steps))),
        "save_freq": int(save_freq),
        "use_policy_training_preset": True,
        "wandb": {
            "enable": False,
            "project": f"every_embodied_{policy_type}",
            "entity": None,
            "disable_artifact": True,
        },
    }


def write_training_config(name, config):
    config_dir = OUTPUT_ROOT / "configs"
    config_dir.mkdir(parents=True, exist_ok=True)
    path = config_dir / f"{name}.yaml"
    path.write_text(
        yaml.safe_dump(config, allow_unicode=True, sort_keys=False),
        encoding="utf-8",
    )
    print("已写出配置：", path)
    return path


def training_command(config_path):
    return [
        sys.executable,
        str(PROJECT_ROOT / "train_model.py"),
        "--config_path",
        str(Path(config_path)),
    ]


def run_training(config_path, enabled=False):
    require_project_layout()
    command = training_command(config_path)
    print("$", shlex.join(command))
    if not enabled:
        print("当前只预览命令。确认配置后，把对应 RUN_* 开关改为 True。")
        return None
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{PROJECT_ROOT}:{env.get('PYTHONPATH', '')}"
    return subprocess.run(command, cwd=PROJECT_ROOT, env=env, check=True)


def show_rocm_resources():
    command = ["rocm-smi", "--showuse", "--showmemuse", "--showtemp"]
    print("$", shlex.join(command))
    if shutil.which(command[0]) is None:
        print("未找到 rocm-smi。确认当前机器是否安装 ROCm。")
        return
    result = subprocess.run(command, check=False, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip(), file=sys.stderr)


## Checkpoint 1：设置采集目录与安全开关


In [4]:
DATASET_REPO_ID = os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language_local")
COLLECTION_ROOT = Path(
    os.environ.get("COLLECTION_ROOT", DATA_ROOT / "omy_pnp_language")
).expanduser()
NUM_DEMOS = int(os.environ.get("NUM_DEMOS", "20"))
SEED_START = int(os.environ.get("SEED_START", "0"))
INSTRUCTIONS = [
    "Place the red mug on the plate.",
    "Place the blue mug on the plate.",
]
RUN_INTERACTIVE_COLLECTION = False
OVERWRITE_DATASET = False
RECORD_FOUR_VIEW_VIDEO = True
COLLECTION_VIDEO_DIR = Path(
    os.environ.get("COLLECTION_VIDEO_DIR", OUTPUT_ROOT / "collection_four_view")
).expanduser()

print("dataset repo_id =", DATASET_REPO_ID)
print("collection root =", COLLECTION_ROOT)
print("num demos =", NUM_DEMOS)
print("instructions =", INSTRUCTIONS)
print("four-view videos =", COLLECTION_VIDEO_DIR)


dataset repo_id = datawhale_eai_pnp_language_local
collection root = $TRAIN_DATA_ROOT
num demos = 20
instructions = ['Place the red mug on the plate.', 'Place the blue mug on the plate.']
four-view videos = $OUTPUT_ROOT/collection_four_view


`COLLECTION_ROOT` 应放在大容量数据盘。默认不开启采集，也不会删除已有目录；只有明确把 `OVERWRITE_DATASET=True` 后才允许覆盖。20 条可以跑通语言条件小实验，想做位置泛化时建议采 30–50 条高质量轨迹，并让红杯、蓝杯和初始位置都得到覆盖。


## Checkpoint 2：检查项目、显示和数据 schema


In [5]:
require_project_layout()
print("MuJoCo scene =", PROJECT_ROOT / "asset" / "example_scene_y2.xml")
print("DISPLAY =", os.environ.get("DISPLAY"))
if not os.environ.get("DISPLAY"):
    print("当前没有 DISPLAY。交互采集需要本地桌面、远程桌面或可用的 X11 会话。")

FEATURES = {
    "observation.image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.wrist_image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["state"],
    },
    "action": {
        "dtype": "float32",
        "shape": (7,),
        "names": ["action"],
    },
    "obj_init": {
        "dtype": "float32",
        "shape": (9,),
        "names": [
            "red_x", "red_y", "red_z",
            "blue_x", "blue_y", "blue_z",
            "plate_x", "plate_y", "plate_z",
        ],
    },
}
md_table(
    ["字段", "shape", "作用"],
    [
        ("observation.image", "256x256x3", "固定相机"),
        ("observation.wrist_image", "256x256x3", "腕部相机"),
        ("observation.state", "6", "当前 6 关节状态"),
        ("action", "7", "下一步 6 关节目标 + 夹爪"),
        ("obj_init", "9", "三件物体初始 xyz，仅用于回放/审计"),
    ],
)


MuJoCo scene = $PROJECT_ROOT/asset/example_scene_y2.xml
DISPLAY = None
当前没有 DISPLAY。交互采集需要本地桌面、远程桌面或可用的 X11 会话。
| 字段 | shape | 作用 |
| --- | --- | --- |
| observation.image | 256x256x3 | 固定相机 |
| observation.wrist_image | 256x256x3 | 腕部相机 |
| observation.state | 6 | 当前 6 关节状态 |
| action | 7 | 下一步 6 关节目标 + 夹爪 |
| obj_init | 9 | 三件物体初始 xyz，仅用于回放/审计 |


## Checkpoint 3：定义严格成功判定


In [6]:
import numpy as np


def collection_snapshot(env, initial_target_z, initial_plate_pos, max_target_lift, max_lifted_run):
    target_pos = np.asarray(env.env.get_p_body(env.obj_target), dtype=np.float64)
    plate_pos = np.asarray(env.env.get_p_body("body_obj_plate_11"), dtype=np.float64)
    target_R = np.asarray(env.env.get_R_body(env.obj_target), dtype=np.float64)
    legacy_success = bool(env.check_success())
    upright_cos = float(target_R[2, 2])
    z_gap = abs(float(target_pos[2] - plate_pos[2]))
    plate_xy_displacement = float(
        np.linalg.norm(plate_pos[:2] - np.asarray(initial_plate_pos)[:2])
    )
    placement_candidate = bool(
        legacy_success
        and max_target_lift >= 0.03
        and max_lifted_run >= 3
        and upright_cos >= 0.7
        and z_gap < 0.08
        and plate_xy_displacement < 0.05
    )
    return {
        "legacy_success": legacy_success,
        "placement_candidate": placement_candidate,
        "xy_dist": float(np.linalg.norm(target_pos[:2] - plate_pos[:2])),
        "plate_z_gap": z_gap,
        "plate_xy_displacement": plate_xy_displacement,
        "max_target_lift": float(max_target_lift),
        "max_lifted_run": int(max_lifted_run),
        "upright_cos": upright_cos,
    }


旧 `check_success()` 只看终态几何关系，杯子被推到盘子附近也可能触发。这里额外要求杯子至少抬升 3 cm、连续保持 3 个控制步、终态保持直立，并且杯子高度与盘子相符。只有 `physical_success=True` 才写入 episode。


## Checkpoint 4：创建或加载 LeRobot 数据集


In [7]:
def create_or_load_dataset():
    from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

    COLLECTION_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if COLLECTION_ROOT.exists() and OVERWRITE_DATASET:
        shutil.rmtree(COLLECTION_ROOT)
    if COLLECTION_ROOT.exists():
        print("继续写入已有数据集：", COLLECTION_ROOT)
        return LeRobotDataset(DATASET_REPO_ID, root=COLLECTION_ROOT)
    print("创建新数据集：", COLLECTION_ROOT)
    return LeRobotDataset.create(
        repo_id=DATASET_REPO_ID,
        root=COLLECTION_ROOT,
        robot_type="omy",
        fps=20,
        features=FEATURES,
        image_writer_threads=10,
        image_writer_processes=0,
    )


## Checkpoint 5：键盘采集完整循环


In [8]:
def collect_demonstrations():
    import random
    import numpy as np
    from PIL import Image
    from mujoco_env.y_env2 import SimpleEnv2

    dataset = create_or_load_dataset()
    xml_path = PROJECT_ROOT / "asset" / "example_scene_y2.xml"
    COLLECTION_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

    try:
        import cv2
    except ImportError:
        cv2 = None
        if RECORD_FOUR_VIEW_VIDEO:
            print("未安装 OpenCV，数据仍会采集，但跳过四视角 MP4。")

    video_writer = None
    video_tmp_path = None

    def open_episode_video(episode_id):
        nonlocal video_writer, video_tmp_path
        if not RECORD_FOUR_VIEW_VIDEO or cv2 is None:
            return
        video_tmp_path = COLLECTION_VIDEO_DIR / f"episode_{episode_id:03d}.recording.mp4"
        video_writer = cv2.VideoWriter(
            str(video_tmp_path),
            cv2.VideoWriter_fourcc(*"mp4v"),
            20.0,
            (640, 480),
        )
        if not video_writer.isOpened():
            raise RuntimeError(f"无法创建四视角视频：{video_tmp_path}")

    def write_episode_video(env, agent_image, wrist_image):
        if video_writer is None:
            return
        views = [
            ("Agent", agent_image),
            ("Egocentric", wrist_image),
            ("Top", env.env.get_fixed_cam_rgb(cam_name="topview")),
            ("Side", env.env.get_fixed_cam_rgb(cam_name="sideview")),
        ]
        panels = []
        for label, image in views:
            panel = cv2.resize(np.asarray(image), (320, 240), interpolation=cv2.INTER_AREA)
            panel = cv2.cvtColor(panel, cv2.COLOR_RGB2BGR)
            cv2.rectangle(panel, (0, 0), (150, 28), (20, 20, 20), thickness=-1)
            cv2.putText(
                panel, label, (8, 20), cv2.FONT_HERSHEY_SIMPLEX,
                0.55, (255, 255, 255), 1, cv2.LINE_AA,
            )
            panels.append(panel)
        video_writer.write(np.vstack([np.hstack(panels[:2]), np.hstack(panels[2:])]))

    def close_episode_video(episode_id, keep):
        nonlocal video_writer, video_tmp_path
        if video_writer is not None:
            video_writer.release()
        if video_tmp_path is not None and video_tmp_path.exists():
            if keep:
                final_path = COLLECTION_VIDEO_DIR / f"episode_{episode_id:03d}_success.mp4"
                video_tmp_path.replace(final_path)
                print("saved four-view video:", final_path)
            else:
                video_tmp_path.unlink()
        video_writer = None
        video_tmp_path = None

    def reset_episode(env, episode_id):
        seed = SEED_START + episode_id
        np.random.seed(seed)
        random.seed(seed)
        env.reset(seed=seed)
        instruction = INSTRUCTIONS[episode_id % len(INSTRUCTIONS)]
        env.set_instruction(instruction)
        initial_z = float(env.env.get_p_body(env.obj_target)[2])
        initial_plate_pos = np.asarray(
            env.env.get_p_body("body_obj_plate_11"), dtype=np.float64
        ).copy()
        print(f"episode={episode_id} seed={seed} task={instruction}")
        return initial_z, initial_plate_pos

    np.random.seed(SEED_START)
    random.seed(SEED_START)
    env = SimpleEnv2(str(xml_path), seed=SEED_START, state_type="joint_angle")
    episode_id = 0
    record_flag = False
    initial_target_z, initial_plate_pos = reset_episode(env, episode_id)
    max_target_lift = 0.0
    lifted_run = 0
    max_lifted_run = 0
    stable_place_steps = 0

    try:
        while env.env.is_viewer_alive() and episode_id < NUM_DEMOS:
            env.step_env()
            if not env.env.loop_every(HZ=20):
                continue

            target_z = float(env.env.get_p_body(env.obj_target)[2])
            lift = target_z - initial_target_z
            max_target_lift = max(max_target_lift, lift)
            lifted_run = lifted_run + 1 if lift >= 0.03 else 0
            max_lifted_run = max(max_lifted_run, lifted_run)
            status = collection_snapshot(
                env,
                initial_target_z,
                initial_plate_pos,
                max_target_lift,
                max_lifted_run,
            )
            stable_place_steps = stable_place_steps + 1 if status["placement_candidate"] else 0
            status["stable_place_steps"] = stable_place_steps
            status["physical_success"] = bool(stable_place_steps >= 5)

            if record_flag and status["physical_success"]:
                dataset.save_episode()
                close_episode_video(episode_id, keep=True)
                print("saved:", json.dumps(status, ensure_ascii=False))
                episode_id += 1
                record_flag = False
                if episode_id >= NUM_DEMOS:
                    break
                initial_target_z, initial_plate_pos = reset_episode(env, episode_id)
                max_target_lift = 0.0
                lifted_run = 0
                max_lifted_run = 0
                stable_place_steps = 0
                continue

            teleop_delta, reset = env.teleop_robot()
            if reset:
                dataset.clear_episode_buffer()
                close_episode_video(episode_id, keep=False)
                record_flag = False
                initial_target_z, initial_plate_pos = reset_episode(env, episode_id)
                max_target_lift = 0.0
                lifted_run = 0
                max_lifted_run = 0
                stable_place_steps = 0
                continue

            if not record_flag and np.any(np.abs(teleop_delta) > 1e-8):
                record_flag = True
                open_episode_video(episode_id)
                print("Start recording")

            agent_image, wrist_image = env.grab_image()
            if record_flag:
                write_episode_video(env, agent_image, wrist_image)
            state = env.get_joint_state()[:6].astype(np.float32)
            env.step(teleop_delta)
            target_action = env.q[:7].astype(np.float32)

            if record_flag:
                dataset.add_frame(
                    {
                        "observation.image": np.asarray(Image.fromarray(agent_image).resize((256, 256))),
                        "observation.wrist_image": np.asarray(Image.fromarray(wrist_image).resize((256, 256))),
                        "observation.state": state,
                        "action": target_action,
                        "obj_init": np.asarray(env.obj_init_pose, dtype=np.float32),
                    },
                    task=env.instruction,
                )
            env.render(teleop=True, idx=episode_id)
    finally:
        close_episode_video(episode_id, keep=False)
        env.env.close_viewer()
        shutil.rmtree(dataset.root / "images", ignore_errors=True)
    print(f"采集完成：{episode_id}/{NUM_DEMOS}")
    return dataset


if RUN_INTERACTIVE_COLLECTION:
    dataset = collect_demonstrations()
else:
    print("采集默认关闭。确认 DISPLAY、路径和覆盖开关后，将 RUN_INTERACTIVE_COLLECTION 改为 True。")


采集默认关闭。确认 DISPLAY、路径和覆盖开关后，将 RUN_INTERACTIVE_COLLECTION 改为 True。


键位与上游教程一致：`W/A/S/D` 控制平面移动，`R/F` 控制高度，`Q/E` 与方向键控制姿态，空格切换夹爪，`Z` 丢弃当前失败回合。每次成功保存后会重新关闭记录开关，避免把 reset 过程混进下一条 episode。模型训练仍只使用固定相机和腕部相机；Top/Side 视角写入独立 MP4，专门用于人工复核，不偷偷扩充模型输入。


## Checkpoint 6：采集后审计


In [9]:
info_path = COLLECTION_ROOT / "meta" / "info.json"
if info_path.exists():
    info, tasks = dataset_report(COLLECTION_ROOT)
    episodes_path = COLLECTION_ROOT / "meta" / "episodes.jsonl"
    episodes = [
        json.loads(line)
        for line in episodes_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    task_counts = {}
    for episode in episodes:
        task = episode.get("tasks", [""])[0]
        task_counts[task] = task_counts.get(task, 0) + 1
    md_table(["指令", "episode 数"], sorted(task_counts.items()))
else:
    print("还没有可审计的数据：", COLLECTION_ROOT)


| 数据项 | 读取结果 |
| --- | --- |
| repo_id |  |
| episodes | 20 |
| frames | 2621 |
| fps | 20 |
| state shape | [6] |
| action shape | [7] |
| tasks | Place the blue mug on the plate. / Place the red mug on the plate. |
| 指令 | episode 数 |
| --- | --- |
| Place the blue mug on the plate. | 10 |
| Place the red mug on the plate. | 10 |


数据表通过后，再打开上游 `6.visualize_data.ipynb` 随机回放若干 episode。至少检查图像、state/action shape、夹爪开闭时序、是否真的抬起，以及红蓝杯指令是否平衡。完成这些检查后再进入 08–10 的训练 Notebook。


## Checkpoint 7：查看已经跑过的数据集实测结果


In [10]:
snapshot = json.loads(
    (ASSET_DIR / "collection_dataset_snapshot.json").read_text(encoding="utf-8")
)
rows = [
    ("repo_id", snapshot["dataset_repo_id"]),
    ("episodes", snapshot["total_episodes"]),
    ("frames", snapshot["total_frames"]),
    ("fps", snapshot["fps"]),
    ("state / action", f'{snapshot["state_shape"]} / {snapshot["action_shape"]}'),
    ("模型输入相机", " / ".join(snapshot["stored_cameras"])),
    ("复核视频视角", " / ".join(snapshot["recorder_views"])),
]
md_table(["数据项", "AMD 实测值"], rows)
print("平均每条轨迹帧数 =", round(snapshot["total_frames"] / snapshot["total_episodes"], 1))


| 数据项 | AMD 实测值 |
| --- | --- |
| repo_id | datawhale_eai_pnp_language |
| episodes | 20 |
| frames | 2621 |
| fps | 20 |
| state / action | [6] / [7] |
| 模型输入相机 | observation.image / observation.wrist_image |
| 复核视频视角 | Agent / Egocentric / Top / Side |
平均每条轨迹帧数 = 131.1


## Checkpoint 8：四视角采集/回放样例


下面的视频是在 AMD 设备上用与键盘采集相同的四视角 recorder 实际录制的严格成功策略回放。它用于先检查画面布局、接触、抬升、搬运和释放是否清楚；它不是人工键盘示教。真正采集时，将 `RUN_INTERACTIVE_COLLECTION=True`，每条成功示教会在 `COLLECTION_VIDEO_DIR` 生成同样布局的 MP4。

            ![MuJoCo 四视角严格成功序列](../assets/pnp_four_view_strict_success_sequence.jpg)

            <video controls width="960" src="../assets/pnp_four_view_strict_success.mp4"></video>


## Checkpoint 9：确认环境 seed 真的改变物体位置


In [11]:
source_path = PROJECT_ROOT / "mujoco_env" / "y_env2.py"
source = source_path.read_text(encoding="utf-8")
bad_pattern = "np.random.seed(seed=0)"
good_pattern = "np.random.seed(seed=seed)"
print("seed source =", source_path)
print("发现旧的固定 seed 写法：", bad_pattern in source)
print("发现修正后的 seed 写法：", good_pattern in source)
if bad_pattern in source:
    raise RuntimeError(
        "当前 y_env2.py 会把所有环境固定成 seed 0。先修复它，再采集位置随机化数据。"
    )


seed source = $PROJECT_ROOT/mujoco_env/y_env2.py
发现旧的固定 seed 写法： False
发现修正后的 seed 写法： True


这个检查很重要。旧实现曾把任意传入 seed 都改成 `0`，导致多 seed 评估只改变了策略采样，杯子和盘子位置并没有真正变化。修复后要重新采 30–50 条覆盖不同位置的轨迹；旧固定场景数据仍可用于链路 smoke，但不能作为空间泛化训练集。
